In [8]:
import duckdb


In [9]:
con = duckdb.connect(database= 'dados_duckdb.db', read_only= False)

In [18]:
df = con.execute("""
            SELECT * 
            FROM (
            SELECT *, ROW_NUMBER () OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row 
            FROM bronze_z0019
            WHERE data_ingestao >= '2025-01-11'    
            ) WHERE ROW = 1
                """).fetchdf()
df.head(10)
         


,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-14 15:02:34.262019,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-03-14 15:02:34.262019,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-14 15:09:14.698371,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-14 15:09:14.698371,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-03-14 15:09:14.698371,1


In [37]:
df_final = df.drop (columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"nome"})
df_final = df_final.rename(columns={"WERKS":"categoria"})
df_final = df_final.rename(columns={"MAINS":"fornecedor"})
df_final = df_final.rename(columns={"LABST":"preço"})
df_final.head(10)

,id,nome,categoria,fornecedor,preço
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [38]:
df_final.dtypes

id            str
nome          str
categoria     str
fornecedor    str
preço         str
dtype: object

In [ ]:
df2 = df_final
df2 = df2.astype(
    {
        'id': int,
        'nome': str,
        'categoria': str,
        'fornecedor': int,
        'preço': float
    }

)
df2.dtypes


id              int64
nome              str
categoria         str
fornecedor      int64
preço         float64
dtype: object

In [41]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT,
            nome TEXT,
            categoria TEXT,
            fornecedor BIGINT,
            preço FLOAT
            )  
""")

In [42]:
df2.head(10)

,id,nome,categoria,fornecedor,preço
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [44]:
con.execute("INSERT INTO produtos SELECT * from df2")

In [45]:
df_resultado = con.execute("select * from produtos").fetch_df()
df_resultado.head(10)

,id,nome,categoria,fornecedor,preço
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [46]:
con.close()
